# Step 20.3 — Save Configuration and Metadata

## 数组保存以后，怎样保留“这些数组该如何解释”的说明？

20.1 保存模型权重，20.2 保存 Adam 状态树。本节补上第三部分：**configuration（配置）与 metadata（元数据）**。重点不是再写一遍 Transformer，而是跟踪同一套字符映射、架构、进度与文件引用如何进入 JSON，之后如何读回。

本节主线分为：架构字段 → 三字符映射 → 规范化表示与指纹 → 进度/seed → 文件路径 → 保存与后续核对。所有代码仍沿用原有协议；旧 MiniGPT 完整定义移到明确标记的可选附录，不是本节新增知识。

**准备独立工作副本。** 在仓库根目录先同步：

```bash
git pull
```

再复制到时间戳新文件名，不覆盖已有笔记：

```bash
mkdir -p notebooks/work && cp -n notebooks/89_step20_3_save_configuration_metadata.ipynb "notebooks/work/89_step20_3_save_configuration_metadata_$(date +%Y%m%d_%H%M%S)_work.ipynb"
```

以后只编辑自己的 work 副本；同秒重试也由 `cp -n` 避免覆盖。本母版所有代码单元未执行、无输出；本次不读取语料，不保存、加载或恢复真实 checkpoint，不运行 MLX、训练或验证。下文具体数值与路径均为纸面说明，不是运行记录。环境沿用 Python 3.12 与 MLX v0.32.2。

## 1. 三类产物不是同一个东西

| 产物 | 回答的问题 | 本身不能回答的问题 |
|---|---|---|
| 20.1 的模型权重 | 每个参数数组当前是什么数值？ | 怎样构造结构、token ID 对应哪个字符？ |
| 20.2 的 NPZ + optimizer manifest | Adam 的数组叶子与树路径怎样保存？ | 这份状态与哪一份模型、哪一次更新配对？ |
| 本节的 run metadata JSON | 用什么配置与映射解释产物、记录了哪些进度和引用？ | 记录是否真实、文件是否来自同一训练边界？ |

**JSON（JavaScript Object Notation）**能保存字典、列表、字符串、数字、布尔值和空值等基础数据，不能直接替代 MLX 数组格式。这里的 `manifest` 是“说明清单”，不是模型本身；20.2 的 optimizer manifest 则专门解释优化器状态树，两个 JSON 各有职责。

下面导入的 `hashlib` 用于计算指纹，`json` 用于文本序列化，`Path` 用于路径，`datetime/timezone` 用于 UTC 时间戳。保留原有 MLX 导入，以使附录完整定义仍可独立使用；主线不会构造模型或调用这些保存函数。

In [ ]:
import hashlib
import json
from datetime import datetime, timezone
from pathlib import Path

import mlx.core as mx
import mlx.nn as nn

## 2. Architecture：先写清数组的解释方式

**Architecture（架构）**记录模型构造参数。沿用 width $C=64$、heads $H=4$，每头宽度是：

$$D=\frac{C}{H}=\frac{64}{4}=16.$$

| 配置键 | 含义 | 为什么恢复时要保留？ |
|---|---|---|
| `dims` | 隐藏宽度 $C$ | 决定投影、归一化等参数的维度 |
| `num_heads` | 头数 $H$ | 决定怎样分组 Q/K/V；只看权重 shape 不一定能发现头数变化 |
| `num_blocks` | Block 层数 | 决定有多少组层参数 |
| `max_seq_len` | 位置表容量 | 决定 learned position embedding 的长度 |
| `vocab_size` | 真实映射的大小 $V$ | token embedding 和 LM head 的行数依赖它 |

本模型 token embedding 与 LM head 权重都含有 $(V,C)$ 的对应维度。即使两个词表大小相同、数组 shape 相同，各行代表的字符仍可能不同，下一节会用三字符例子说明。

下面 `vocab_size=None` 是“尚未提供”的占位，不是假设真实 $V$。`SEEDS` 也只是原有候选初始 seed 配置，不是训练完成证据。

In [ ]:
MODEL_CONFIG = {
    "dims": 64,
    "num_heads": 4,
    "num_blocks": 2,
    "max_seq_len": 64,
    "vocab_size": None,  # Supply from the real token-ID mapping.
}
SEEDS = {"model": 2028, "train": 2026, "validation": 2027}

### 配置检查：验证什么，又没有验证什么？

`validate_model_config(config, vocab_size)` 接收配置字典和已经确认的真实词表大小。按函数顺序读：

1. `required` 列出四个架构字段；`any(...)` 只要发现一个不是正的 Python `int` 就拒绝。使用 `type(...) is int` 也避免把 `True` 当成整数 1。
2. `%` 是取余；`dims % num_heads == 0` 才能把宽度平均分给各头。
3. 单独要求传入的词表大小为正整数。
4. `dict(config)` 创建浅副本，在副本里填入真实 `vocab_size`，返回新字典，不改调用者原字典。

它不读取权重、不检查模型代码版本，也不证明这些配置确实用于某个已有模型。保存函数后面用 `len(stoi)` 提供 $V$，而不是从文件名猜测。

In [ ]:
def validate_model_config(config, vocab_size):
    required = ("dims", "num_heads", "num_blocks", "max_seq_len")
    if any(type(config.get(key)) is not int or config[key] < 1 for key in required):
        raise ValueError("Architecture dimensions must be positive integers.")
    if config["dims"] % config["num_heads"] != 0:
        raise ValueError("dims must be divisible by num_heads.")
    if type(vocab_size) is not int or vocab_size < 1:
        raise ValueError("vocab_size must come from the actual tokenizer mapping.")
    result = dict(config)
    result["vocab_size"] = vocab_size
    return result

## 3. 用三个字符追踪 tokenizer：从映射到可重建记录

**Tokenizer（分词/编码规则）**在本项目是字符级映射：`stoi` 表示 string-to-integer，字符变 ID；反向表用于把 ID 变回字符。以下只是纸面微型词表，不替代真实训练词表：

| 字符 | 原 `stoi` 的 ID | 按 ID 排列的 `itos_by_id` 位置 |
|---|---:|---|
| `a` | 0 | 第 0 项 `a` |
| `b` | 1 | 第 1 项 `b` |
| `猫` | 2 | 第 2 项 `猫` |

因此 `stoi = {"a": 0, "b": 1, "猫": 2}`，反向列表是 `["a", "b", "猫"]`。字符串“猫a”编码为 `[2,0]`，再查列表就还原为“猫a”。词表的角色是赋予数组行和 logits 类别以语义。

### 同样大小，为什么仍然可能完全错配？

现在只把 ID 0/1 交换：`{"a":1,"b":0,"猫":2}`，反向列表变为 `["b","a","猫"]`。两套词表都有 $V=3$，token embedding 的 shape 都可以是 $(3,64)$，但同一个 ID 0 从 `a` 变成了 `b`。沿用旧权重却更换映射，会把旧的第 0 行解释成另一个字符。**shape 匹配是必要检查之一，不是语义匹配证明。**

所以记录必须包含完整映射，而不只是 `vocab_size` 或一串 hash。Hash 只能用于核对，不能从摘要反推出原词表。

### 保存映射函数：先检查 ID，再按 ID 放回字符

`tokenizer_metadata(stoi)` 的前半段逐步做以下事情：

- `ids = list(stoi.values())` 取得所有 ID；先检查每项是 Python 整数。
- `sorted(ids) == list(range(len(ids)))` 要求恰好是 `0,1,...,V-1`，同时拒绝重复或缺失 ID。例子 `[0,2,2]` 不等于 `[0,1,2]`。
- `chars_by_id = [""] * len(stoi)` 先建长度 $V$ 的列表。循环中 `chars_by_id[token_id] = character` 按 ID 填槽，不依赖字典的插入顺序。
- 每个 key 必须是长度为 1 的 Python 字符串。本实现没有额外做 Unicode 归一化，也不是按“人眼看到的字形数量”来验证。

对纸面词表，最后必定得到 `["a","b","猫"]`，即使输入字典恰好先写 `猫` 再写 `a`，也不会改变按 ID 的次序。

### Canonical representation：先约定字节，再计算 hash

**Canonical（规范化表示）**在这里指本课程明确约定的序列化方案，不是任意 JSON 文件都天然有唯一写法。`enumerate(chars_by_id)` 依次给出 `(0,"a")`、`(1,"b")`、`(2,"猫")`；列表推导式交换每对的排列，生成 `[字符,ID]`。最终 JSON 文本应为：

```text
[["a",0],["b",1],["猫",2]]
```

- `ensure_ascii=False` 保留中文字符，不把它写成 Unicode 转义序列。
- `separators=(",", ":")` 去掉分隔符处多余空格；这里主体是列表，关键是逗号后的空格被省去。
- `.encode("utf-8")` 把文本转换为字节；SHA-256 处理字节，不处理抽象的 Python 字典。
- `hashlib.sha256(canonical).hexdigest()` 返回十六进制摘要。我们不伪造一串未计算的摘要作为实测输出。

交换 `a/b` 的 ID 后，规范化文本变为 `[["b",0],["a",1],["猫",2]]`，所编码的映射不同；摘要核对用于发现这种变化。SHA-256 不是数学上的无碰撞编码，也不是加密或签名，但可以作为实用的一致性检查。

**当前 `stoi_sha256` 只覆盖按 ID 排列的字符-ID 对及其上述字节表示。** 它不覆盖整个 metadata JSON，也不覆盖权重、seed、进度、架构或语料。返回值中的 `type`、`vocab_size`、`stoi`、`itos_by_id` 还要在恢复时各自核对。原文件中关于“排序后 stoi”的概括，应准确理解为按 ID 次序，而不是按字符字母顺序排序。

In [ ]:
def tokenizer_metadata(stoi):
    if not isinstance(stoi, dict) or not stoi:
        raise ValueError("Provide the actual non-empty character-to-ID mapping.")
    ids = list(stoi.values())
    if any(type(token_id) is not int for token_id in ids):
        raise TypeError("All token IDs must be Python integers.")
    if sorted(ids) != list(range(len(ids))):
        raise ValueError("Token IDs must be contiguous from 0 to vocab_size - 1.")

    # Store the complete mapping, not just a fingerprint: the hash validates,
    # while these fields allow the same mapping to be reconstructed later.
    chars_by_id = [""] * len(stoi)
    for character, token_id in stoi.items():
        if not isinstance(character, str) or len(character) != 1:
            raise ValueError("This character tokenizer expects single-character string keys.")
        chars_by_id[token_id] = character

    canonical = json.dumps(
        [[character, token_id] for token_id, character in enumerate(chars_by_id)],
        ensure_ascii=False,
        separators=(",", ":"),
    ).encode("utf-8")
    return {
        "type": "character",
        "vocab_size": len(stoi),
        "stoi": dict(stoi),
        "itos_by_id": chars_by_id,
        "stoi_sha256": hashlib.sha256(canonical).hexdigest(),
    }

## 4. 从记录重建，而不是重新读取语料猜词表

现在反过来沿 `reconstruct_stoi_from_tokenizer_metadata` 阅读：

| 步骤 | 三字符例子的纸面结果 | 拒绝什么？ |
|---|---|---|
| 检查 `type` | `character` | 其他 tokenizer 类型 |
| 读取列表与长度 | `["a","b","猫"]`，长度 3 | 不是列表、长度与记录的 $V$ 不符 |
| 检查单字符 | 三项都是长度 1 的字符串 | 空字符串、多字符字符串等 |
| 用 `enumerate` 建字典 | `{"a":0,"b":1,"猫":2}` | 若列表含重复字符，字典长度变短，从而拒绝 |
| 重新生成规范字节、算摘要 | 同一字符-ID 表示 | 摘要与 `stoi_sha256` 不符 |
| 比较保存的 `stoi` | 与重建字典一致 | 两份映射互相矛盾 |

为什么同时保存两种映射？`itos_by_id` 明确列表位置的 ID 意义，`stoi` 明确编码方向，二者互检可以发现记录冲突。不过，如果错误映射及其摘要被一起一致地更改，这些内部检查仍可能通过；它们不证明权重实际训练时使用的就是这份映射。还需要可靠的原始记录与配对流程。

注意，下面只定义重建函数，不调用它；也不会访问 corpus、模型或权重文件。

In [ ]:
def reconstruct_stoi_from_tokenizer_metadata(tokenizer):
    if tokenizer.get("type") != "character":
        raise ValueError("Unsupported tokenizer type.")
    chars_by_id = tokenizer.get("itos_by_id")
    if not isinstance(chars_by_id, list) or len(chars_by_id) != tokenizer.get("vocab_size"):
        raise ValueError("Saved character table does not match vocab_size.")
    if any(not isinstance(ch, str) or len(ch) != 1 for ch in chars_by_id):
        raise ValueError("Invalid saved character table.")
    stoi = {character: token_id for token_id, character in enumerate(chars_by_id)}
    if len(stoi) != len(chars_by_id):
        raise ValueError("Saved character table contains duplicate characters.")
    # Rebuild and verify the fingerprint without consulting a corpus.
    canonical = json.dumps(
        [[character, token_id] for token_id, character in enumerate(chars_by_id)],
        ensure_ascii=False,
        separators=(",", ":"),
    ).encode("utf-8")
    digest = hashlib.sha256(canonical).hexdigest()
    if digest != tokenizer.get("stoi_sha256"):
        raise ValueError("Tokenizer mapping fingerprint mismatch.")
    if tokenizer.get("stoi") != stoi:
        raise ValueError("Saved stoi and itos_by_id disagree.")
    return stoi

## 5. Seed、随机状态与进度分别记录什么？

**Random seed（随机种子）**是随机序列的起点输入；**RNG state（随机数生成器状态）**是推进到某一时刻后继续生成所需的状态。纸面上想象同一个 seed 已经依次产生了第 1、2、3 组随机选择：重新用初始 seed 只会回到起点，并不自动知道下一次应该接第 4 组。

本协议保存 `initial_seeds`，并保留 `rng_state_saved` 标记，默认 `False`。这两个字段都不包含完整 RNG 状态。即使调用者把布尔标记改为 `True`，函数也只是检查其类型，不会替你保存随机状态或验证相关文件。不能仅凭这个标记声称精确续训。

### 更新编号也要有统一含义

以下数字仅用于说明版本关系，不是本项目的真实训练结果：计划 100 次更新，已经完成第 7 次；第 7 次 update 使用的是更新前参数（已完成 6 次时的参数）计算 loss，验证则可能在更新后进行。

| 字段 | 纸面例子 | 解释 |
|---|---:|---|
| `planned_updates` | 100 | 计划总数，不是已执行次数 |
| `completed_updates` | 7 | 已确实完成的 optimizer update 次数 |
| `last_train_loss_update` | 6 | 最近记录训练 loss 所用的参数版本，按“已完成更新数”标记 |
| `last_validation_update` | 7 | 最近一次验证时模型已完成的更新数 |

`None` 表示相应记录未提供，例如尚未验证；不是把“不知道”写成 0。Warm-up 若真实改变了参数，是否计入计数必须与 optimizer step 保持一致的约定。

`validate_progress` 检查必需计数为非负 Python 整数、completed 不超过 planned；可选两项接受 `None` 或非负整数，返回字典副本。**它没有读取 Adam step，也没有校验可选 loss/validation 编号是否真的对应保存的参数。** 例如错误的但非负编号仍可能通过；真实性来自运行记录，后续还要核对恢复后的 Adam step。

In [ ]:
def validate_progress(progress):
    required_ints = ("planned_updates", "completed_updates")
    for key in required_ints:
        value = progress.get(key)
        if type(value) is not int or value < 0:
            raise ValueError(f"{key} must be a non-negative Python int.")
    if progress["completed_updates"] > progress["planned_updates"]:
        raise ValueError("completed_updates cannot exceed planned_updates.")
    for key in ("last_train_loss_update", "last_validation_update"):
        value = progress.get(key)
        if value is not None and (type(value) is not int or value < 0):
            raise ValueError(f"{key} must be None or a non-negative Python int.")
    return dict(progress)

## 6. Artifact：文件内容摘要与 tokenizer 摘要不是同一个范围

**Artifact（产物文件）**在这里是已经存在的权重、优化器数组或优化器 manifest。`artifact_metadata(path, artifact_root)` 要求调用者明确选定文件与共同根目录，不会猜“最新文件”。

按函数逐段读：

1. `Path(...).resolve()` 将路径解析为绝对路径；`is_file()` 要求目标是现有文件。
2. `path.relative_to(root)` 得到相对于共同根的路径，若文件在根目录之外则拒绝。`as_posix()` 用 `/` 表示相对路径。
3. `read_bytes()` 读取文件的实际字节来算 SHA-256；`stat().st_size` 记录字节数。这里只有函数定义，当前没有发生读取。
4. 返回的字典只包含 `relative_path`、`size_bytes`、`sha256`，不复制或移动文件。

与 tokenizer 的规范化摘要不同，这个文件摘要覆盖**整个文件的原始字节**。例如 optimizer JSON 仅改变缩进，内容语义可能一样，文件字节摘要仍会改变。记录不包含文件本体，也不证明其来源、训练正确性或同一 run 配对。

此实现一次读取整个文件，且读取和 `stat()` 是先后操作，不是原子快照。未来只能对已经保存完毕、不再被并发修改的产物采集记录；本节没有提供锁、事务或大文件流式 hashing。

In [ ]:
def artifact_metadata(path, artifact_root):
    path = Path(path).resolve()
    root = Path(artifact_root).resolve()
    if not path.is_file():
        raise FileNotFoundError(f"Expected existing artifact file: {path}")
    try:
        relative_path = path.relative_to(root)
    except ValueError as exc:
        raise ValueError("Artifact must be inside the chosen artifact root.") from exc
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    return {
        "relative_path": relative_path.as_posix(),
        "size_bytes": path.stat().st_size,
        "sha256": digest,
    }

# Definition only. This function is not called in the published master.

### 明确两种位置：metadata_path 不等于 artifact_root

下面是与 20.1、20.2、20.3 当前函数相容的**假想目录**。假设项目是 `/home/learner/mlx-minigpt-from-scratch`，20.1 接收项目根，20.2 和本节保存函数的 `output_root` 都传项目的 `results`。时间戳仅作占位示例：

```text
/home/learner/mlx-minigpt-from-scratch/results/       ← artifact_root
├── model_weights/
│   └── minigpt_weights_20261101T010000Z/
│       └── model.safetensors
├── optimizer_states/
│   └── adam_state_20261101T010000Z/
│       ├── optimizer_state.npz
│       └── optimizer_state.json
└── run_metadata/
    └── minigpt_run_20261101T010001Z/
        └── run_metadata.json                     ← metadata_path
```

权重记录的 `relative_path` 应是 `model_weights/minigpt_weights_20261101T010000Z/model.safetensors`；优化器数组与 manifest 对应 `optimizer_states/adam_state_20261101T010000Z/` 下的两个文件。字典的三个键必须与后续解析器相配：`weights`、`optimizer_arrays`、`optimizer_manifest`。

未来恢复时，`metadata_path` 用来打开那份说明 JSON；`artifact_root` 用来拼接每个记录的相对路径。**不能把 `metadata_path.parent` 当成 artifact_root**，否则会在 `run_metadata/minigpt_run_.../model_weights/...` 下错误寻找文件。移动整棵 `results` 目录到新电脑后，可以显式选择新根；相对文件布局和字节仍须匹配。

这些字符串只是路径说明，不是本机已有文件，不应直接复制为真实 checkpoint 路径。20.1/20.2 的独立时间戳也不能证明来自同一训练边界；必须先确认原始权重与 optimizer 是同一组，再为其采集记录。

## 7. 保存外层 JSON：按四段读，而不是直接运行整个函数

下面保留原完整保存函数，新增逻辑已经分段说明，不要求重新学附录里的模型。

**第一段：检查调用者传入的数据。** 由 `len(stoi)` 得到 $V$，分别调用架构、tokenizer 与 progress 检查函数；检查 seeds 是整数值字典、`rng_state_saved` 是布尔值。它不检查 seeds 的名字是否齐全，也不保存 RNG state。

**第二段：建立新目录。** `output_root/run_metadata` 可复用；带 UTC 时间戳的 `run_dir` 使用 `exist_ok=False` 拒绝重名，同一秒重复调用可能失败而不是覆盖。`run_label` 应使用可信的简单目录名；当前接口未把任意外部标签当作安全输入做专项验证。

**第三段：组装 manifest。** `format_version=1` 是本节外层 metadata 协议，不是 20.2 optimizer manifest 的版本 2。`architecture` 与 `tokenizer` 是解释数据；`training_progress` 与 `initial_seeds` 是记录；`artifacts` 是文件引用字典。

特别注意：代码只有 `"artifacts": dict(artifacts)`，**不会在这里自动调用 `artifact_metadata` 或重查引用文件**。调用者必须事先为三个真实、稳定、相互配对的文件生成记录，并在恢复时重新验证。元数据函数也没有记录或校验 Adam 全部超参数、软件环境、数据版本与 sampler 状态；不能把当前字段范围夸大为完整 checkpoint。

**第四段：独占创建文件。** `open("x", encoding="utf-8")` 拒绝覆盖已有同名 JSON，`json.dump(..., ensure_ascii=False, indent=2)` 写入易读格式。这里的 `indent=2` 与前面用于映射 hash 的紧凑表示不是同一个用途；`stoi_sha256` 不随整个文件的缩进变化。

拒绝覆盖并不等于“原子事务”：目录创建后，后续序列化或磁盘写入失败可能留下目录或不完整文件，函数没有回滚。返回 `manifest_path` 仅表明这次写入流程返回了一个路径，不是已经验证训练恢复正确。

In [ ]:
def save_run_metadata_to_new_directory(
    output_root, model_config, stoi, seeds, progress, artifacts,
    *, rng_state_saved=False, run_label="minigpt_run",
):
    vocab_size = len(stoi)
    config = validate_model_config(model_config, vocab_size)
    tokenizer = tokenizer_metadata(stoi)
    checked_progress = validate_progress(progress)
    if not isinstance(seeds, dict) or any(type(v) is not int for v in seeds.values()):
        raise ValueError("seeds must be a mapping of names to Python integer seed values.")
    if type(rng_state_saved) is not bool:
        raise TypeError("rng_state_saved must be bool.")

    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    root = Path(output_root) / "run_metadata"
    root.mkdir(parents=True, exist_ok=True)
    run_dir = root / f"{run_label}_{stamp}"
    run_dir.mkdir(exist_ok=False)

    manifest = {
        "format_version": 1,
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "architecture": config,
        "tokenizer": tokenizer,  # includes reconstructable stoi and itos_by_id, plus SHA-256
        "initial_seeds": dict(seeds),
        "rng_state_saved": rng_state_saved,
        "training_progress": checked_progress,
        "artifacts": dict(artifacts),
        "note": "Metadata only; not a full training checkpoint or proof of exact resumption.",
    }
    manifest_path = run_dir / "run_metadata.json"
    with manifest_path.open("x", encoding="utf-8") as f:
        json.dump(manifest, f, ensure_ascii=False, indent=2)
    return manifest_path

# Definition only. No metadata file is written during material preparation.

## 8. 接回后续恢复：每层检查的结论都有限

这里只阅读已存在的 20.4/20.5 接口来确定本节输出怎样被消费，不在本轮执行或推进那些课节：

1. `load_checkpoint_metadata(metadata_path)` 读外层 JSON，检查版本、架构字段，并重建字符表；核对 architecture 的 $V$ 与重建映射长度。
2. `resolve_checkpoint_artifacts(artifact_root, metadata)` 将三个相对路径接到**同一个原始约定的根**下，再检查是否越界、是否存在、大小及 SHA-256。
3. 根据保存的架构构造模型，严格加载权重；按原运行的 Adam 配置创建目标并恢复状态树。当前本节 metadata 并未存 Adam 超参数，不能假定别处的默认值必然适用。
4. 核对 Adam step 与 `completed_updates`，再按 20.5 比较参数、状态与小输入 logits。未执行的检查不能写成 PASS。

| 已知证据 | 可以说明 | 不能单独说明 |
|---|---|---|
| `stoi`、反向表与指纹互相一致 | 保存记录内部一致，可重建同一映射 | 权重原先一定使用此映射 |
| 文件大小与 SHA-256 匹配 | 所读文件与记录的字节一致 | 三份文件来自同一 run、模型训练正确 |
| 架构和参数 shape 匹配 | 结构上满足这些检查 | 每个 token 行的语义必然正确 |
| Adam step 与进度相同 | 这两个计数一致 | sampler/RNG/数据顺序完整恢复 |
| seed 已记录 | 知道最初如何初始化随机序列 | 已恢复到中途随机位置 |

本节没有签名、可信来源验证或完整恢复证明。要追求逐步相同的后续训练，还需实际保存/重建适用的 RNG state、sampler 游标、数据与 split、学习率调度和环境等；只写一个布尔值或 seed 不足以补齐这些状态。

### 自查（不附答案代码）

1. 三字符词表仅改变字典插入顺序与改变 ID 分配，有何区别？哪一种改变规范化表示？
2. 记录一个 hash 为什么不能代替 `itos_by_id`？内部检查通过为什么仍不能证明与权重来源相同？
3. 根据目录图，说明把 `metadata_path.parent` 传成 `artifact_root` 会拼出什么错误位置。
4. `completed_updates=7`、`rng_state_saved=True` 只是在 JSON 里出现，为什么还不能宣称精确续训？

本节新增知识是“解释配置、可重建映射、字节范围、明确路径和证据边界”。未来只有真实对象与文件已经准备好，才选择调用保存函数；不需要为完成本节而先运行一轮训练或伪造 checkpoint。

## 可选附录 · 既有完整 MiniGPT 定义，不是 metadata 新代码

以下原样保留 Step 15.4 的 MHA、MLP、TransformerBlock 和 MiniGPT，供未来需要独立模型定义的工作副本查阅；主线 metadata 函数不实例化或依赖这些类。**读懂本节保存协议，不需要重新逐行学习这些长代码。** 它们不是“无需理解直接运行”的新实验，也没有自动 forward、训练或保存调用。

若未来需要使用附录类，先具备上方 imports，再按 MHA → MLP/Block → MiniGPT 的顺序定义。这里只移动完整定义到附录，没有改架构、参数命名、forward 或数值计算。新建模型仍只是初始化模型，不能当成已训练 checkpoint。

### A. Causal Multi-Head Attention：旧实现

输入/输出均为 $(B,T,C)$；Q/K/V 分头为 $(B,H,T,D)$，局部因果 mask 只在 forward 内创建。下方是原定义的复用，不是本节新增的序列化算法。

In [ ]:
import math
import mlx.core as mx
import mlx.nn as nn


class CausalMultiHeadAttention(nn.Module):
    def __init__(self, dims: int, num_heads: int):
        super().__init__()

        if dims <= 0:
            raise ValueError('dims must be positive')
        if num_heads <= 0:
            raise ValueError('num_heads must be positive')
        if dims % num_heads != 0:
            raise ValueError('dims must be divisible by num_heads')

        self.dims = dims
        self.num_heads = num_heads
        self.head_dim = dims // num_heads
        self.scale = 1.0 / math.sqrt(self.head_dim)  # Python float, not trainable

        self.qkv_proj = nn.Linear(
            input_dims=dims,
            output_dims=3 * dims,
            bias=False,
        )
        self.c_proj = nn.Linear(
            input_dims=dims,
            output_dims=dims,
            bias=False,
        )

    def __call__(self, x: mx.array) -> mx.array:
        if len(x.shape) != 3:
            raise ValueError('attention input must have shape (B,T,C)')

        B, T, C = x.shape
        if C != self.dims:
            raise ValueError('attention input width must equal dims')

        H = self.num_heads
        D = self.head_dim

        qkv = self.qkv_proj(x)  # (B,T,3C)
        q, k, v = mx.split(qkv, indices_or_sections=3, axis=-1)

        q = mx.reshape(q, (B, T, H, D))
        k = mx.reshape(k, (B, T, H, D))
        v = mx.reshape(v, (B, T, H, D))

        q = mx.swapaxes(q, 1, 2)  # (B,H,T,D)
        k = mx.swapaxes(k, 1, 2)
        v = mx.swapaxes(v, 1, 2)

        scores = mx.matmul(
            q,
            mx.swapaxes(k, -1, -2),
        ) * self.scale  # (B,H,T,T)

        # IMPORTANT: structural mask is local to forward.
        # It is NOT stored as self.causal_mask.
        future = mx.triu(mx.ones((T, T)), k=1) > 0
        mask = mx.where(
            future,
            mx.array(float('-inf'), dtype=scores.dtype),
            mx.array(0.0, dtype=scores.dtype),
        )

        weights = mx.softmax(scores + mask, axis=-1)
        head_out = mx.matmul(weights, v)  # (B,H,T,D)

        token_major = mx.swapaxes(head_out, 1, 2)  # (B,T,H,D)
        merged = mx.reshape(token_major, (B, T, C))
        return self.c_proj(merged)  # (B,T,C)

### B. Feed-Forward MLP 与 TransformerBlock：旧实现

保留两层带 bias 的 MLP、GELU，以及两条 Pre-Norm RMSNorm 残差路径；这些结构为什么存在已在模型课节解释。本节只要求记录与它们相容的构造配置，不改变其逻辑。

In [ ]:
class FeedForwardMLP(nn.Module):
    def __init__(self, dims: int):
        super().__init__()
        self.fc1 = nn.Linear(dims, 4 * dims, bias=True)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(4 * dims, dims, bias=True)

    def __call__(self, x: mx.array) -> mx.array:
        return self.fc2(self.act(self.fc1(x)))


class TransformerBlock(nn.Module):
    def __init__(self, dims: int, num_heads: int):
        super().__init__()
        self.norm1 = nn.RMSNorm(dims=dims)
        self.attn = CausalMultiHeadAttention(dims=dims, num_heads=num_heads)
        self.norm2 = nn.RMSNorm(dims=dims)
        self.mlp = FeedForwardMLP(dims=dims)

    def __call__(self, x: mx.array) -> mx.array:
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x


# Definitions only. This master notebook does not execute them.

### C. MiniGPT：旧完整模型汇总

token/position embedding → 若干独立 Block → final RMSNorm → untied LM head，输出 raw logits $(B,T,V)$。完整定义用于未来独立使用，不读取 metadata、不加载权重，也不等于已完成恢复。

In [ ]:
class MiniGPT(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        dims: int,
        num_heads: int,
        num_blocks: int,
        max_seq_len: int,
    ):
        super().__init__()

        if vocab_size <= 0:
            raise ValueError('vocab_size must be positive')
        if dims <= 0:
            raise ValueError('dims must be positive')
        if num_heads <= 0:
            raise ValueError('num_heads must be positive')
        if num_blocks <= 0:
            raise ValueError('num_blocks must be positive')
        if max_seq_len <= 0:
            raise ValueError('max_seq_len must be positive')
        if dims % num_heads != 0:
            raise ValueError('dims must be divisible by num_heads')

        # Python metadata: these are not trainable mx.array parameters.
        self.vocab_size = vocab_size
        self.dims = dims
        self.max_seq_len = max_seq_len

        # Learned lookup tables.
        self.token_embedding = nn.Embedding(
            num_embeddings=vocab_size,
            dims=dims,
        )
        self.position_embedding = nn.Embedding(
            num_embeddings=max_seq_len,
            dims=dims,
        )

        # A list of distinct Module instances; MLX recursively tracks them.
        self.blocks = [
            TransformerBlock(
                dims=dims,
                num_heads=num_heads,
            )
            for _ in range(num_blocks)
        ]

        self.final_norm = nn.RMSNorm(dims=dims)
        self.lm_head = nn.Linear(
            input_dims=dims,
            output_dims=vocab_size,
            bias=False,
        )

    def __call__(self, token_ids: mx.array) -> mx.array:
        if len(token_ids.shape) != 2:
            raise ValueError('token_ids must have shape (B,T)')

        B, T = token_ids.shape

        if T < 1:
            raise ValueError('sequence length T must be at least 1')
        if T > self.max_seq_len:
            raise ValueError(
                f'sequence length {T} exceeds max_seq_len {self.max_seq_len}'
            )

        # Forward-local structural indices: not Module parameters.
        position_ids = mx.arange(T)  # (T,)

        token_features = self.token_embedding(token_ids)        # (B,T,C)
        position_features = self.position_embedding(position_ids)  # (T,C)
        x = token_features + position_features                  # (B,T,C)

        for block in self.blocks:
            x = block(x)  # invariant: (B,T,C)

        x = self.final_norm(x)  # (B,T,C)
        logits = self.lm_head(x)  # (B,T,V)

        expected_shape = (B, T, self.vocab_size)
        if logits.shape != expected_shape:
            raise RuntimeError(
                f'logits shape {logits.shape} != expected {expected_shape}'
            )

        return logits


# Definition only. No MiniGPT forward is executed in this master notebook.

## 参考、版本与本次材料边界

- [20.1 模型权重](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/cdd8508458a8aa202baeddb5affd31021095c978/notebooks/87_step20_1_save_model_weights.ipynb)：独立权重目录。
- [20.2 优化器状态](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/cdd8508458a8aa202baeddb5affd31021095c978/notebooks/88_step20_2_save_optimizer_state.ipynb)：NPZ 与状态 manifest。
- [20.4 已有恢复消费者](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/cdd8508458a8aa202baeddb5affd31021095c978/notebooks/90_step20_4_resume_training.ipynb)、[20.5 已有验证边界](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/cdd8508458a8aa202baeddb5affd31021095c978/notebooks/91_step20_5_resume_verification.ipynb)。链接用于本节输入/输出契约核对，不表示本轮修改或执行这些课节。

本次仅修订 20.3 的教学组织和说明，保留原 Python 函数/类、配置、协议版本与执行路径。历史修正计数与历史静态验收记录不因重新讲解而归零，也不把旧验收当作本次修订已验收。本次没有执行 Notebook、MLX、保存函数、checkpoint、训练、恢复或测试；静态检查不等于运行通过。完成本节修订后等待独立验收，不自动进入下一节。